# C02–M3–L1 — Build an AI Icebreaker Bot with IBM Granite 3.0 & LlamaIndex (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/hengkuankian/learning_ibm_rag_agentic/tree/main/Course2_Build_RAG_Applications/Build_IE_Bot_w_Granite_and_LlamaIndex)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- Project venv + `requirements.txt` (llama-index 0.11 + IBM integrations), the old Groq/HF `get_llm()` setup and `llm_factory.py` removed → Cell 1 = minimal Gemini install + `llama-index-core`, `llama-index-llms-google-genai`, `llama-index-embeddings-google-genai`, `gradio` (one pip line); Cell 2 = universal Gemini init.
- Whole project kept as `%%writefile` cells in the original layout (`config.py`, `modules/…`, `main.py`, `app.py`) inside `icebreaker/`.
- `modules/llm_interface.py` (same function names): `WatsonxEmbeddings` (slate-125m) → `GoogleGenAIEmbedding("gemini-embedding-2")`, `WatsonxLLM` (Granite 3.2 8B) → `GoogleGenAI("gemini-3.8-flash")`; the 30-line `CustomLLM` wrapper is gone.
- `config.py`: watsonx URL/project id and `MODEL_MAP`/`EMBEDDING_MAP` removed; app dropdown = `gemini-3.8-flash` · `gemini-3.5-flash-lite` · `gemini-2.5-flash`.
- `app.py` chat history switched to the Gradio 6 *messages* format (`[user, bot]` pairs no longer work); earlier fixes kept (`main.py` imports `config`, `launch(share=True, debug=True)`, ProxyCurl → mock profile JSON).
- CLI chat fed with piped questions; Colab exercises at the end (own profile, icebreakers, chunk/top-k sweep, Gemini model comparison, chat engine with memory).

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai gradio

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the `GOOGLE_API_KEY` Colab secret.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
# Gemini 3: no temperature (Google recommends the default 1.0; low values can loop)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

This lab uses **LlamaIndex**, not LangChain. Cell 2's LangChain `llm` is not needed by the project; the scripts build LlamaIndex's own Gemini classes (`GoogleGenAI`, `GoogleGenAIEmbedding`) in `modules/llm_interface.py`, reading `GOOGLE_API_KEY` from the environment.

## Project files
All files go into `icebreaker/`, matching the original repo layout.

### 💡 How to use the ✅ Answer cells
Every exercise is followed by a hidden **✅ Answer** cell (form view). *Run all* stops at the first one on purpose (`Answer not run`), so you try first.
To check your work: double-click the cell to show the code (or just run it), tick **run_answer**, run it. Below each answer, a markdown line says what you should see.
Cells after an answer cell need a manual run (or *Runtime → Run after*).

In [ ]:
import os
os.makedirs('icebreaker/modules', exist_ok=True)
%cd icebreaker

### `config.py` — settings and prompt templates
watsonx model ids → free Gemini model ids (no URL / project id needed). Edit the two templates to change the bot's style.

In [ ]:
%%writefile config.py
"""Configuration settings for the Icebreaker Bot."""

# Colab (Gemini): IBM watsonx.ai URL / project id are not needed — GOOGLE_API_KEY comes from the environment

# Model settings (Colab: watsonx ids → Gemini)
LLM_MODEL_ID = "gemini-3.8-flash"            # was ibm/granite-3-2-8b-instruct
EMBEDDING_MODEL_ID = "gemini-embedding-2"    # was ibm/slate-125m-english-rtrvr-v2

# ProxyCurl API settings
PROXYCURL_API_KEY = ""  # ProxyCurl has shut down (2025) — use the mock data or your own profile JSON

# Mock data URL
MOCK_DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZRe59Y_NJyn3hZgnF1iFYA/linkedin-profile-data.json"

# Query settings
SIMILARITY_TOP_K = 5
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 500
MIN_NEW_TOKENS = 1
TOP_K = 50
TOP_P = 1

# Node settings
CHUNK_SIZE = 500

# LLM prompt templates
INITIAL_FACTS_TEMPLATE = """
You are an AI assistant that provides detailed answers based on the provided context.

Context information is below:

{context_str}

Based on the context provided, list 3 interesting facts about this person's career or education.

Answer in detail, using only the information provided in the context.
"""

USER_QUESTION_TEMPLATE = """
You are an AI assistant that provides detailed answers to questions based on the provided context.

Context information is below:

{context_str}

Question: {query_str}

Answer in full details, using only the information provided in the context. If the answer is not available in the context, say "I don't know. The information is not available on the LinkedIn page."
"""


### `test_config.py` — test your configuration (lab: *Testing Your Configuration*)
Same script as the lab. It only reads `config.py`, so it needs no API key.

In [ ]:
%%writefile test_config.py
import config

print("Initial Facts Template defined:", bool(config.INITIAL_FACTS_TEMPLATE))
print("User Question Template defined:", bool(config.USER_QUESTION_TEMPLATE))
print("Chunk Size:", config.CHUNK_SIZE)
print("Similarity Top K:", config.SIMILARITY_TOP_K)

In [ ]:
!python test_config.py

*Lab Step 5 (optional):* for longer profiles the lab suggests `CHUNK_SIZE = 400` and `SIMILARITY_TOP_K = 7` in `config.py` — practice C3 compares such values.

### `modules/__init__.py`

In [ ]:
%%writefile modules/__init__.py
"""
Icebreaker Bot modules package.

This package contains the following modules:
- data_extraction: Functions for extracting LinkedIn profile data
- data_processing: Functions for processing and indexing LinkedIn profile data
- llm_interface: Functions for interfacing with Gemini LLMs (was IBM watsonx.ai)
- query_engine: Functions for querying indexed LinkedIn profile data
"""

from modules.data_extraction import extract_linkedin_profile
from modules.data_processing import split_profile_data, create_vector_database, verify_embeddings
from modules.llm_interface import create_watsonx_embedding, create_watsonx_llm, change_llm_model
from modules.query_engine import generate_initial_facts, answer_user_query

### 📝 Lab Exercise 1 · Pseudocode for `extract_linkedin_profile`
*From the lab:* “Take a moment to think about how you would implement this function and create the pseudocode for it before moving on to the next step.”
Write your pseudocode (as comments) in the next cell **before** reading `modules/data_extraction.py`.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 1 · extract_linkedin_profile
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

# PSEUDOCODE
# extract_linkedin_profile(url, api_key=None, mock=False):
#   start a timer
#   if mock: GET config.MOCK_DATA_URL (timeout 30 s)
#   else:    require api_key → GET ProxyCurl endpoint with {"url": url, ...} and an
#            "Authorization: Bearer <api_key>" header
#   if status != 200 → log the error, return {}
#   data = response.json()
#   clean it: drop empty values ([], "", None) and unused keys ("people_also_viewed", "certifications")
#   if "groups" exist → drop "profile_pic_url" from each group
#   log the elapsed time, return the cleaned dict   (any exception → log, return {})

# Runnable sketch of the mock branch (same steps, no ProxyCurl):
import time, requests, config
def my_extract(url, api_key=None, mock=True):
    t = time.time()
    r = requests.get(config.MOCK_DATA_URL, timeout=30)
    if r.status_code != 200:
        return {}
    data = {k: v for k, v in r.json().items()
            if v not in ([], "", None) and k not in ("people_also_viewed", "certifications")}
    print(f"loaded in {time.time() - t:.2f}s")
    return data
p = my_extract("https://www.linkedin.com/in/leonkatsnelson/")
print(p.get("full_name"), "|", len(p), "fields |", sorted(p)[:6])

**Expected / what you should see:** `loaded in …s`, then `Leon Katsnelson | N fields | [...]` (the lab's mock profile). Your pseudocode should cover: mock vs API branch, status-code check, JSON cleanup, error → `{}`.

### `modules/data_extraction.py` — get the LinkedIn profile
Unchanged. `mock=True` downloads the lab's sample profile JSON (ProxyCurl is no longer available).

In [ ]:
%%writefile modules/data_extraction.py
"""Module for extracting LinkedIn profile data."""

import time
import requests
import logging
from typing import Dict, Optional, Any

import config

logger = logging.getLogger(__name__)

def extract_linkedin_profile(
    linkedin_profile_url: str, 
    api_key: Optional[str] = None, 
    mock: bool = False
) -> Dict[str, Any]:
    """Extract LinkedIn profile data using ProxyCurl API or loads a premade JSON file.
    
    Args:
        linkedin_profile_url: The LinkedIn profile URL to extract data from.
        api_key: ProxyCurl API key. Required if mock is False.
        mock: If True, loads mock data from a premade JSON file instead of using the API.
    
    Returns:
        Dictionary containing the LinkedIn profile data.
    """
    start_time = time.time()
    
    try:
        if mock:
            logger.info("Using mock data from a premade JSON file...")
            mock_url = config.MOCK_DATA_URL
            response = requests.get(mock_url, timeout=30)
        else:
            # Ensure API key is provided when mock is False
            if not api_key:
                raise ValueError("ProxyCurl API key is required when mock is set to False.")
            
            logger.info("Starting to extract the LinkedIn profile...")

            # Set up the API endpoint and headers
            api_endpoint = "https://nubela.co/proxycurl/api/v2/linkedin"
            headers = {
                "Authorization": f"Bearer {api_key}"
            }

            # Prepare parameters for the request
            params = {
                "url": linkedin_profile_url,
                "fallback_to_cache": "on-error",
                "use_cache": "if-present",
                "skills": "include",
                "inferred_salary": "include",
                "personal_email": "include",
                "personal_contact_number": "include"
            }

            logger.info(f"Sending API request to ProxyCurl at {time.time() - start_time:.2f} seconds...")

            # Send API request
            response = requests.get(api_endpoint, headers=headers, params=params, timeout=10)
        
        logger.info(f"Received response at {time.time() - start_time:.2f} seconds...")

        # Check if response is successful
        if response.status_code == 200:
            try:
                # Parse the JSON response
                data = response.json()
                
                # Clean the data, remove empty values and unwanted fields
                data = {
                    k: v
                    for k, v in data.items()
                    if v not in ([], "", None) and k not in ["people_also_viewed", "certifications"]
                }

                # Remove profile picture URLs from groups to clean the data
                if data.get("groups"):
                    for group_dict in data.get("groups"):
                        group_dict.pop("profile_pic_url", None)

                return data
            except ValueError as e:
                logger.error(f"Error parsing JSON response: {e}")
                logger.error(f"Response content: {response.text[:200]}...")  # Print first 200 chars
                return {}
        else:
            logger.error(f"Failed to retrieve data. Status code: {response.status_code}")
            logger.error(f"Response: {response.text}")
            return {}
            
    except Exception as e:
        logger.error(f"Error in extract_linkedin_profile: {e}")
        return {}

### 📝 Lab Exercise 2 · Pseudocode for `split_profile_data`, `create_vector_database`, `verify_embeddings`
*From the lab:* “Take a moment to think about how you would implement these functions and create the pseudocode for them before moving on to the next step.”
Write your pseudocode (as comments) in the next cell **before** reading `modules/data_processing.py`.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 2 · split / index / verify
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

# PSEUDOCODE
# split_profile_data(profile):
#   text = json.dumps(profile)                      # whole profile as one string
#   doc  = Document(text=text)
#   nodes = SentenceSplitter(chunk_size=config.CHUNK_SIZE).get_nodes_from_documents([doc])
#   return nodes                                    (exception → log, return [])
# create_vector_database(nodes):
#   embed_model = create_watsonx_embedding()
#   index = VectorStoreIndex(nodes=nodes, embed_model=embed_model, show_progress=False)
#   return index                                    (exception → log, return None)
# verify_embeddings(index):
#   for every node in index.vector_store data: embedding must exist and not be empty
#   log the ids that are missing → return True only if none are missing

import json, requests, config
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
prof = requests.get(config.MOCK_DATA_URL, timeout=30).json()
my_nodes = SentenceSplitter(chunk_size=config.CHUNK_SIZE).get_nodes_from_documents([Document(text=json.dumps(prof))])
my_idx = VectorStoreIndex(nodes=my_nodes, embed_model=GoogleGenAIEmbedding(model_name=config.EMBEDDING_MODEL_ID))
emb = my_idx.vector_store.data.embedding_dict
missing = [nid for nid in emb if not emb[nid]]
print(len(my_nodes), "nodes ·", len(emb), "embeddings · missing:", missing)

**Expected / what you should see:** about 10–20 nodes (chunk size 500), the same number of embeddings, `missing: []`.

### `modules/data_processing.py` — split into nodes and index
Unchanged: JSON → `Document` → `SentenceSplitter(chunk_size=500)` nodes → `VectorStoreIndex`.

In [ ]:
%%writefile modules/data_processing.py
"""Module for processing LinkedIn profile data."""

import json
import logging
from typing import Dict, List, Any, Optional

from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

from modules.llm_interface import create_watsonx_embedding
import config

logger = logging.getLogger(__name__)

def split_profile_data(profile_data: Dict[str, Any]) -> List:
    """Splits the LinkedIn profile JSON data into nodes.
    
    Args:
        profile_data: LinkedIn profile data dictionary.
        
    Returns:
        List of document nodes.
    """
    try:
        # Convert the profile data to a JSON string
        profile_json = json.dumps(profile_data)

        # Create a Document object from the JSON string
        document = Document(text=profile_json)

        # Split the document into nodes using SentenceSplitter
        splitter = SentenceSplitter(chunk_size=config.CHUNK_SIZE)
        nodes = splitter.get_nodes_from_documents([document])
        
        logger.info(f"Created {len(nodes)} nodes from profile data")
        return nodes
    except Exception as e:
        logger.error(f"Error in split_profile_data: {e}")
        return []

def create_vector_database(nodes: List) -> Optional[VectorStoreIndex]:
    """Stores the document chunks (nodes) in a vector database.
    
    Args:
        nodes: List of document nodes to be indexed.
        
    Returns:
        VectorStoreIndex or None if indexing fails.
    """
    try:
        # Get the embedding model
        embedding_model = create_watsonx_embedding()

        # Create a VectorStoreIndex from the nodes
        index = VectorStoreIndex(
            nodes=nodes,
            embed_model=embedding_model,
            show_progress=False
        )
        
        logger.info("Vector database created successfully")
        return index
    except Exception as e:
        logger.error(f"Error in create_vector_database: {e}")
        return None

def verify_embeddings(index: VectorStoreIndex) -> bool:
    """Verify that all nodes have been properly embedded.
    
    Args:
        index: VectorStoreIndex to verify.
        
    Returns:
        True if all embeddings are valid, False otherwise.
    """
    try:
        vector_store = index._storage_context.vector_store
        node_ids = list(index.index_struct.nodes_dict.keys())
        missing_embeddings = False

        for node_id in node_ids:
            embedding = vector_store.get(node_id)
            if embedding is None:
                logger.warning(f"Node ID {node_id} has a None embedding.")
                missing_embeddings = True
        
        if missing_embeddings:
            logger.warning("Some node embeddings are missing")
            return False
        else:
            logger.info("All node embeddings are valid")
            return True
    except Exception as e:
        logger.error(f"Error in verify_embeddings: {e}")
        return False

### 📝 Lab Exercise 3 · Pseudocode for `create_watsonx_embedding`, `create_watsonx_llm`, `change_llm_model`
*From the lab:* “Take a moment to think about how you would implement these functions and create the pseudocode for them before moving on to the next step.”
Write your pseudocode (as comments) in the next cell **before** reading `modules/llm_interface.py`.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 3 · embedding + LLM factory
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

# PSEUDOCODE
# create_watsonx_embedding(): return an embedding model for config.EMBEDDING_MODEL_ID
#   (IBM: WatsonxEmbeddings(model_id, url, project_id, truncate_input_tokens=3)
#    Colab: GoogleGenAIEmbedding(model_name=config.EMBEDDING_MODEL_ID))
# create_watsonx_llm(temperature, max_new_tokens, decoding_method):
#   greedy → temperature 0
#   build the LLM for config.LLM_MODEL_ID with those generation settings
#   (Colab/Gemini 3: no temperature, thinking_level="low", max_output_tokens ≥ 1024)
# change_llm_model(new_id): config.LLM_MODEL_ID = new_id   (next create_watsonx_llm() uses it)

import asyncio
try:
    asyncio.get_running_loop()
    import nest_asyncio
    _orig_run = asyncio.run
    nest_asyncio.apply()
    _nested_run = asyncio.run
    def _run(main, *, debug=None, loop_factory=None):
        # nest_asyncio's run() has no loop_factory → uvicorn (Gradio launch) would crash
        if loop_factory is not None:
            return _orig_run(main, debug=debug, loop_factory=loop_factory)
        return _nested_run(main, debug=debug)
    asyncio.run = _run
except RuntimeError:
    pass
import config
from google.genai import types
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
def my_llm(max_new_tokens=500):
    return GoogleGenAI(model=config.LLM_MODEL_ID, max_tokens=max(max_new_tokens, 1024),
                       generation_config=types.GenerateContentConfig(
                           max_output_tokens=max(max_new_tokens, 1024),
                           thinking_config=types.ThinkingConfig(thinking_level="low")))
print(len(GoogleGenAIEmbedding(model_name=config.EMBEDDING_MODEL_ID).get_text_embedding("hello")), "dims")
print(my_llm().complete("Say hello in 3 words.").text)

**Expected / what you should see:** `3072 dims` with `gemini-embedding-2` (offline fake: 64), then a short greeting from `gemini-3.8-flash`.

### `modules/llm_interface.py` — the only rewritten module
watsonx → LlamaIndex's Gemini integrations, with the same function names (swap rule: `llama_index.llms.google_genai.GoogleGenAI` + `llama_index.embeddings.google_genai.GoogleGenAIEmbedding`).

In [ ]:
%%writefile modules/llm_interface.py
"""Module for interfacing with LLMs — Colab version (IBM watsonx.ai → Google Gemini).

Same function names as the IBM lab, so the rest of the project is unchanged:
- create_watsonx_embedding() → LlamaIndex GoogleGenAIEmbedding (gemini-embedding-2)
- create_watsonx_llm()       → LlamaIndex GoogleGenAI (gemini-3.8-flash)
"""

import asyncio
import logging

# from llama_index.embeddings.ibm import WatsonxEmbeddings                     ← IBM originals
# from llama_index.llms.ibm import WatsonxLLM
# from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
from google.genai import types
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI

import config

logger = logging.getLogger(__name__)

# Colab fix: LlamaIndex's GoogleGenAI.chat()/complete() call asyncio.run(), which fails inside
# a notebook ("asyncio.run() cannot be called from a running event loop"). nest_asyncio
# (installed with llama-index-core) allows it. Only applied when a loop is running (notebook),
# so `python main.py` / `python app.py` are unchanged.
try:
    asyncio.get_running_loop()
    import nest_asyncio
    _orig_run = asyncio.run
    nest_asyncio.apply()
    _nested_run = asyncio.run
    def _run(main, *, debug=None, loop_factory=None):
        # nest_asyncio's run() has no loop_factory → uvicorn (Gradio launch) would crash
        if loop_factory is not None:
            return _orig_run(main, debug=debug, loop_factory=loop_factory)
        return _nested_run(main, debug=debug)
    asyncio.run = _run
except RuntimeError:
    pass


def create_watsonx_embedding() -> GoogleGenAIEmbedding:
    """Creates the embedding model for vector representation (was IBM watsonx slate)."""
    embedding = GoogleGenAIEmbedding(model_name=config.EMBEDDING_MODEL_ID)
    logger.info(f"Created embedding model: {config.EMBEDDING_MODEL_ID}")
    return embedding


def create_watsonx_llm(
    temperature: float = config.TEMPERATURE,
    max_new_tokens: int = config.MAX_NEW_TOKENS,
    decoding_method: str = "sample"
) -> GoogleGenAI:
    """Creates the LLM for generating responses (was IBM watsonx WatsonxLLM).

    decoding_method="greedy" forces temperature 0, like watsonx (ignored on Gemini 3, see below).
    """
    if decoding_method == "greedy":
        temperature = 0.0
    max_out = max(max_new_tokens, 1024)          # Gemini counts thinking tokens in max_output_tokens
    if config.LLM_MODEL_ID.startswith("gemini-3"):
        # Gemini 3: Google recommends the default temperature (1.0); low values can cause looping.
        # thinking_level="low" leaves the token budget for the answer.
        gen_config = types.GenerateContentConfig(
            max_output_tokens=max_out,
            thinking_config=types.ThinkingConfig(thinking_level="low"),
        )
    else:
        # Gemini 2.5: temperature is fine; cap thinking with a budget
        gen_config = types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_out,
            thinking_config=types.ThinkingConfig(thinking_budget=512),
        )
    llm = GoogleGenAI(
        model=config.LLM_MODEL_ID,
        max_tokens=max_out,
        generation_config=gen_config,   # with generation_config, the temperature/max_tokens args are not sent
    )
    logger.info(f"Created LLM: {config.LLM_MODEL_ID}")
    return llm


def change_llm_model(new_model_id: str) -> None:
    """Change the LLM model to use.

    Args:
        new_model_id: New LLM model ID to use.
    """
    global config
    config.LLM_MODEL_ID = new_model_id
    logger.info(f"Changed LLM model to: {new_model_id}")


### 📝 Lab Exercise 4 · Pseudocode for `generate_initial_facts`, `answer_user_query`
*From the lab:* “Take a moment to think about how you would implement these functions and create the pseudocode for them before moving on to the next step.”
Write your pseudocode (as comments) in the next cell **before** reading `modules/query_engine.py`.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 4 · facts + Q&A
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

# PSEUDOCODE
# generate_initial_facts(index):
#   llm = create_watsonx_llm(temperature=0.0, max_new_tokens=500)
#   prompt = PromptTemplate(config.INITIAL_FACTS_TEMPLATE)            # uses {context_str}
#   qe = index.as_query_engine(similarity_top_k=config.SIMILARITY_TOP_K, llm=llm, text_qa_template=prompt)
#   return qe.query("Provide three interesting facts …").response   (error → message string)
# answer_user_query(index, question):
#   llm = create_watsonx_llm(max_new_tokens=250, decoding_method="greedy")
#   prompt = PromptTemplate(config.USER_QUESTION_TEMPLATE)            # {context_str} + {query_str}
#   qe = index.as_query_engine(... same ..., text_qa_template=prompt)
#   return qe.query(question)          # a Response object → caller uses .response

from llama_index.core import PromptTemplate
import importlib.util   # load the file directly: modules/__init__.py imports query_engine, which isn't written yet
_spec = importlib.util.spec_from_file_location("llm_interface", "modules/llm_interface.py")
_li = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_li)
create_watsonx_llm = _li.create_watsonx_llm
qe = my_idx.as_query_engine(llm=create_watsonx_llm(max_new_tokens=500), similarity_top_k=config.SIMILARITY_TOP_K,
                            text_qa_template=PromptTemplate(config.INITIAL_FACTS_TEMPLATE))
print(qe.query("Provide three interesting facts about this person's career or education.").response)
qa_engine = my_idx.as_query_engine(llm=create_watsonx_llm(max_new_tokens=250, decoding_method="greedy"),
                                   similarity_top_k=config.SIMILARITY_TOP_K,
                                   text_qa_template=PromptTemplate(config.USER_QUESTION_TEMPLATE))
r = qa_engine.query("What is their favourite food?")
print(type(r).__name__, "→", r.response)

**Expected / what you should see:** 3 numbered facts from the mock profile, then `Response → I don't know. The information is not available on the LinkedIn page.` (needs `my_idx` from the Exercise 2 answer).

### `modules/query_engine.py` — facts + Q&A
Unchanged: query engines with the custom `text_qa_template` prompts.

In [ ]:
%%writefile modules/query_engine.py
"""Module for querying indexed LinkedIn profile data."""

import logging
from typing import Any, Dict, Optional

from llama_index.core import VectorStoreIndex, PromptTemplate

from modules.llm_interface import create_watsonx_llm
from llama_index.core.base.response.schema import Response
import config

logger = logging.getLogger(__name__)

def generate_initial_facts(index: VectorStoreIndex) -> str:
    """Generates interesting facts about the person's career or education.
    
    Args:
        index: VectorStoreIndex containing the LinkedIn profile data.
        
    Returns:
        String containing interesting facts about the person.
    """
    try:
        # Create LLM for generating facts
        watsonx_llm = create_watsonx_llm(
            temperature=0.0,
            max_new_tokens=500,
            decoding_method="sample"
        )
        
        # Create prompt template
        facts_prompt = PromptTemplate(template=config.INITIAL_FACTS_TEMPLATE)
        
        # Create query engine
        query_engine = index.as_query_engine(
            streaming=False,
            similarity_top_k=config.SIMILARITY_TOP_K,
            llm=watsonx_llm,
            text_qa_template=facts_prompt
        )
        
        # Execute the query
        query = "Provide three interesting facts about this person's career or education."
        response = query_engine.query(query)
        
        # Return the facts
        return response.response
    except Exception as e:
        logger.error(f"Error in generate_initial_facts: {e}")
        return "Failed to generate initial facts."

def answer_user_query(index: VectorStoreIndex, user_query: str) -> Any:
    """Answers the user's question using the vector database and the LLM.
    
    Args:
        index: VectorStoreIndex containing the LinkedIn profile data.
        user_query: The user's question.
        
    Returns:
        Response object containing the answer to the user's question.
    """
    try:
        # Create LLM for answering questions
        watsonx_llm = create_watsonx_llm(
            temperature=0.0,
            max_new_tokens=250,
            decoding_method="greedy"
        )
        
        # Create prompt template
        question_prompt = PromptTemplate(template=config.USER_QUESTION_TEMPLATE)
        
        # Retrieve relevant nodes
        base_retriever = index.as_retriever(similarity_top_k=config.SIMILARITY_TOP_K)
        source_nodes = base_retriever.retrieve(user_query)
        
        # Build context string
        context_str = "\n\n".join([node.node.get_text() for node in source_nodes])
        
        # Create query engine
        query_engine = index.as_query_engine(
            streaming=False,
            similarity_top_k=config.SIMILARITY_TOP_K,
            llm=watsonx_llm,
            text_qa_template=question_prompt
        )
        
        # Execute the query
        answer = query_engine.query(user_query)
        return answer
    except Exception as e:
        logger.error(f"Error in answer_user_query: {e}")
        return Response(response="Failed to get an answer.")  # Colab fix: callers use .response

### 📝 Lab Exercise 5 · Pseudocode for `process_linkedin`, `chatbot_interface`, `main`
*From the lab:* “Take a moment to think about how you would implement these functions and create the pseudocode for them before moving on to the next step.”
Write your pseudocode (as comments) in the next cell **before** reading `main.py`.

In [ ]:
# Your code here


In [ ]:
# @title ✅ Answer — Lab Exercise 5 · the CLI pipeline
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

# PSEUDOCODE
# process_linkedin(url, api_key, mock):
#   profile = extract_linkedin_profile(url, api_key, mock)   → stop if empty
#   nodes = split_profile_data(profile); index = create_vector_database(nodes)  → stop if None
#   verify_embeddings(index) (warn only); print generate_initial_facts(index)
#   chatbot_interface(index)
# chatbot_interface(index):
#   loop: q = input("You: "); exit on exit/quit/bye; print answer_user_query(index, q).response
# main(): parse --url --api-key --mock --model; ask for the URL only if needed;
#   --model → change_llm_model; process_linkedin(...)

from modules.data_extraction import extract_linkedin_profile
from modules.data_processing import split_profile_data, create_vector_database, verify_embeddings
from modules.query_engine import generate_initial_facts, answer_user_query
def my_pipeline(questions, url="https://www.linkedin.com/in/leonkatsnelson/"):
    idx = create_vector_database(split_profile_data(extract_linkedin_profile(url, mock=True)))
    print("embeddings ok:", verify_embeddings(idx))
    print("\nHere are 3 interesting facts about this person:\n" + generate_initial_facts(idx))
    for q in questions + ["bye"]:              # a list instead of input()
        if q.lower() in ("exit", "quit", "bye"):
            print("Bot: Goodbye!"); break
        print(f"You: {q}\nBot: {answer_user_query(idx, q).response.strip()}\n")
my_pipeline(["What is this person's current job title?", "Where did they get their education?"])

**Expected / what you should see:** `embeddings ok: True`, 3 facts, two `You:`/`Bot:` turns, `Bot: Goodbye!` — the same flow as `python main.py --mock`.

### `main.py` — command-line bot

In [ ]:
%%writefile main.py
"""Main script for running the Icebreaker Bot."""

import sys
import time
import logging
import argparse

from modules.data_extraction import extract_linkedin_profile
from modules.data_processing import split_profile_data, create_vector_database, verify_embeddings
from modules.query_engine import generate_initial_facts, answer_user_query
from typing import Dict, Any, Optional
import config  # Colab fix: main() uses config.PROXYCURL_API_KEY but the original never imported it

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(stream=sys.stdout)
    ]
)

logger = logging.getLogger(__name__)

def process_linkedin(linkedin_url, api_key=None, mock=False):
    """
    Processes a LinkedIn URL, extracts data from the profile, and interacts with the user.

    Args:
        linkedin_url: The LinkedIn profile URL to extract or load mock data from.
        api_key: ProxyCurl API key. Required if mock is False.
        mock: If True, loads mock data from a premade JSON file instead of using the API.
    """
    try:
        # Extract the profile data
        profile_data = extract_linkedin_profile(linkedin_url, api_key, mock=mock)
        
        if not profile_data:
            logger.error("Failed to retrieve profile data.")
            return
        
        # Split the data into nodes
        nodes = split_profile_data(profile_data)
        
        # Store in vector database
        vectordb_index = create_vector_database(nodes)
        
        if not vectordb_index:
            logger.error("Failed to create vector database.")
            return
        
        # Verify embeddings
        if not verify_embeddings(vectordb_index):
            logger.warning("Some embeddings may be missing or invalid.")
        
        # Generate and display the initial facts
        initial_facts = generate_initial_facts(vectordb_index)
        
        print("\nHere are 3 interesting facts about this person:")
        print(initial_facts)
        
        # Start the chatbot interface
        chatbot_interface(vectordb_index)
        
    except Exception as e:
        logger.error(f"Error occurred: {str(e)}")

def chatbot_interface(index):
    """
    Provides a simple chatbot interface for user interaction.
    
    Args:
        index: VectorStoreIndex containing the LinkedIn profile data.
    """
    print("\nYou can now ask more in-depth questions about this person. Type 'exit', 'quit', or 'bye' to quit.")
    
    while True:
        user_query = input("You: ")
        if user_query.lower() in ['exit', 'quit', 'bye']:
            print("Bot: Goodbye!")
            break
        
        print("Bot is typing...", end='')
        sys.stdout.flush()
        time.sleep(1)  # Simulate typing delay
        print('\r', end='')
        
        response = answer_user_query(index, user_query)
        print(f"Bot: {response.response.strip()}\n")

def main():
    """Main function to run the Icebreaker Bot."""
    parser = argparse.ArgumentParser(description='Icebreaker Bot - LinkedIn Profile Analyzer')
    parser.add_argument('--url', type=str, help='LinkedIn profile URL')
    parser.add_argument('--api-key', type=str, help='ProxyCurl API key')
    parser.add_argument('--mock', action='store_true', help='Use mock data instead of API')
    parser.add_argument('--model', type=str, help='LLM model to use (e.g., "gemini-3.8-flash")')
    
    args = parser.parse_args()
    
    # Use command line arguments or prompt user for input
    linkedin_url = args.url or ("" if args.mock else input("Enter LinkedIn profile URL (or press Enter to use mock data): "))  # Colab fix: --mock no longer eats the first piped question
    use_mock = args.mock or not linkedin_url
    
    if args.model:
        from modules.llm_interface import change_llm_model
        change_llm_model(args.model)
    
    api_key = args.api_key or config.PROXYCURL_API_KEY
    
    if not use_mock and not api_key:
        api_key = input("Enter ProxyCurl API key: ")
    
    # Use a default URL for mock data if none provided
    if use_mock and not linkedin_url:
        linkedin_url = "https://www.linkedin.com/in/leonkatsnelson/"
    
    process_linkedin(linkedin_url, api_key, mock=use_mock)

if __name__ == "__main__":
    main()

### `app.py` — Gradio web app

In [ ]:
%%writefile app.py
"""Gradio web interface for the Icebreaker Bot."""

import os
import sys
import logging
import uuid
import gradio as gr

from modules.data_extraction import extract_linkedin_profile
from modules.data_processing import split_profile_data, create_vector_database, verify_embeddings
from modules.llm_interface import change_llm_model
from modules.query_engine import generate_initial_facts, answer_user_query
import config

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(stream=sys.stdout)
    ]
)

logger = logging.getLogger(__name__)

# Dictionary to store active conversations
active_indices = {}

def process_profile(linkedin_url, api_key, use_mock, selected_model):
    """Process a LinkedIn profile and generate initial facts.
    
    Args:
        linkedin_url: LinkedIn profile URL to process.
        api_key: ProxyCurl API key.
        use_mock: Whether to use mock data.
        selected_model: LLM model to use.
        
    Returns:
        Initial facts about the profile and a session ID for this conversation.
    """
    try:
        # Change LLM model if needed
        if selected_model != config.LLM_MODEL_ID:
            change_llm_model(selected_model)
            
        # Use a default URL for mock data if none provided
        if use_mock and not linkedin_url:
            linkedin_url = "https://www.linkedin.com/in/leonkatsnelson/"
            
        # Extract profile data
        profile_data = extract_linkedin_profile(
            linkedin_url,
            api_key if not use_mock else None,
            mock=use_mock
        )
        
        if not profile_data:
            return "Failed to retrieve profile data. Please check the URL or API key.", None
        
        # Split data into nodes
        nodes = split_profile_data(profile_data)
        
        if not nodes:
            return "Failed to process profile data into nodes.", None
        
        # Create vector database
        index = create_vector_database(nodes)
        
        if not index:
            return "Failed to create vector database.", None
        
        # Verify embeddings
        if not verify_embeddings(index):
            logger.warning("Some embeddings may be missing or invalid")
        
        # Generate initial facts
        facts = generate_initial_facts(index)
        
        # Generate a unique session ID
        session_id = str(uuid.uuid4())
        
        # Store the index for this session
        active_indices[session_id] = index
        
        # Return the facts and session ID
        return f"Profile processed successfully!\n\nHere are 3 interesting facts about this person:\n\n{facts}", session_id
    
    except Exception as e:
        logger.error(f"Error in process_profile: {e}")
        return f"Error: {str(e)}", None

def _turn(user_query, answer):
    """Gradio 6 Chatbot uses the "messages" format (the original used [user, bot] pairs)."""
    return [{"role": "user", "content": user_query}, {"role": "assistant", "content": answer}]

def chat_with_profile(session_id, user_query, chat_history):
    """Chat with a processed LinkedIn profile.
    
    Args:
        session_id: Session ID for this conversation.
        user_query: User's question.
        chat_history: Chat history.
        
    Returns:
        Updated chat history.
    """
    if not session_id:
        return chat_history + _turn(user_query, "No profile loaded. Please process a LinkedIn profile first.")
    
    if session_id not in active_indices:
        return chat_history + _turn(user_query, "Session expired. Please process the LinkedIn profile again.")
    
    if not user_query.strip():
        return chat_history
    
    try:
        # Get the index for this session
        index = active_indices[session_id]
        
        # Answer the user's query
        response = answer_user_query(index, user_query)
        
        # Update chat history
        return chat_history + _turn(user_query, response.response)
    
    except Exception as e:
        logger.error(f"Error in chat_with_profile: {e}")
        return chat_history + _turn(user_query, f"Error: {str(e)}")

def create_gradio_interface():
    """Create the Gradio interface for the Icebreaker Bot."""
    # Define available LLM models
    available_models = [
        "gemini-3.8-flash",        # Colab: was ibm/granite-3-2-8b-instruct
        "gemini-3.5-flash-lite",   # Colab: was meta-llama/llama-3-3-70b-instruct slot
        "gemini-2.5-flash",        # Colab: extra option
    ]
    
    with gr.Blocks(title="LinkedIn Icebreaker Bot") as demo:
        gr.Markdown("# LinkedIn Icebreaker Bot")
        gr.Markdown("Generate personalized icebreakers and chat about LinkedIn profiles")
        
        with gr.Tab("Process LinkedIn Profile"):
            with gr.Row():
                with gr.Column():
                    linkedin_url = gr.Textbox(
                        label="LinkedIn Profile URL",
                        placeholder="https://www.linkedin.com/in/username/"
                    )
                    api_key = gr.Textbox(
                        label="ProxyCurl API Key (Leave empty to use mock data)",
                        placeholder="Your ProxyCurl API Key",
                        type="password",
                        value=config.PROXYCURL_API_KEY
                    )
                    use_mock = gr.Checkbox(label="Use Mock Data", value=True)
                    model_dropdown = gr.Dropdown(
                        choices=available_models,
                        label="Select LLM Model",
                        value=config.LLM_MODEL_ID
                    )
                    process_btn = gr.Button("Process Profile")
                
                with gr.Column():
                    result_text = gr.Textbox(label="Initial Facts", lines=10)
                    session_id = gr.Textbox(label="Session ID", visible=False)
            
            process_btn.click(
                fn=process_profile,
                inputs=[linkedin_url, api_key, use_mock, model_dropdown],
                outputs=[result_text, session_id]
            )
        
        with gr.Tab("Chat"):
            gr.Markdown("Chat with the processed LinkedIn profile")
            
            chatbot = gr.Chatbot(height=500)
            chat_input = gr.Textbox(
                label="Ask a question about the profile",
                placeholder="What is this person's current job title?"
            )
            
            chat_btn = gr.Button("Send")
            
            chat_btn.click(
                fn=chat_with_profile,
                inputs=[session_id, chat_input, chatbot],
                outputs=[chatbot]
            )
            
            chat_input.submit(
                fn=chat_with_profile,
                inputs=[session_id, chat_input, chatbot],
                outputs=[chatbot]
            )
    
    return demo

if __name__ == "__main__":
    demo = create_gradio_interface()
    # Launch the Gradio interface
    # You can customize these parameters:
    # - share=True creates a public link you can share with others
    # - server_name and server_port set where the app runs
    # Colab: public *.gradio.live link (was server_name="127.0.0.1", server_port=5000, share=True)
    demo.launch(share=True, debug=True)

## Run step by step
Test each module like the lab does (`python -c ...`), then the full pipeline.

In [ ]:
import sys, importlib, logging; sys.path.insert(0, "."); logging.basicConfig(level=logging.INFO)
import config
from modules.data_extraction import extract_linkedin_profile
profile = extract_linkedin_profile("https://www.linkedin.com/in/leonkatsnelson/", mock=True)
print(list(profile.keys()))

In [ ]:
from modules.data_processing import split_profile_data, create_vector_database, verify_embeddings
nodes = split_profile_data(profile); print(len(nodes), "nodes")
index = create_vector_database(nodes); print("embeddings ok:", verify_embeddings(index))

In [ ]:
from modules.query_engine import generate_initial_facts, answer_user_query
print(generate_initial_facts(index))    # gemini-3.8-flash via LlamaIndex GoogleGenAI
print(answer_user_query(index, "What is this person's current job title?").response)

## Command-line bot (`main.py`)
The questions are piped in; `bye` ends the chat. Add `--model "gemini-3.5-flash-lite"` to switch models.

In [ ]:
!printf "What is this person's current job title?\nWhere did they get their education?\nbye\n" | python main.py --mock

In [ ]:
!printf "What skills do they have related to machine learning?\nbye\n" | python main.py --mock --model "gemini-3.5-flash-lite"

## Web app (`app.py`)
Open the `*.gradio.live` link → **Process Profile** (mock data) → **Chat** tab. Press ⏹ to stop.

In [ ]:
!python app.py

---
# 🧪 Colab practice exercises

### C1 · Your own profile (no ProxyCurl)
Write a profile as JSON (or export your LinkedIn data) and run the same pipeline on it.

In [ ]:
my_profile = {
    "full_name": "Your Name", "headline": "Data Scientist", "city": "Tehran",
    "experiences": [{"title": "ML Engineer", "company": "Acme", "starts_at": {"year": 2022}, "description": "Built RAG chatbots."}],
    "education": [{"school": "University X", "degree_name": "MSc", "field_of_study": "Computer Science"}],
    "skills": ["Python", "LangChain", "LlamaIndex"],
}   # TODO: fill in real details
my_index = create_vector_database(split_profile_data(my_profile))
print(generate_initial_facts(my_index))

In [ ]:
# @title ✅ Answer — C1 · Your own profile
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

my_profile = {
    "full_name": "Sara Ahmadi", "headline": "Data Scientist · RAG & LLM apps", "city": "Tehran", "country_full_name": "Iran",
    "summary": "I build retrieval-augmented chatbots and teach Python.",
    "experiences": [
        {"title": "ML Engineer", "company": "Acme AI", "starts_at": {"year": 2022}, "ends_at": None,
         "description": "Built RAG chatbots with LangChain and LlamaIndex on Gemini."},
        {"title": "Data Analyst", "company": "Blue Bank", "starts_at": {"year": 2019}, "ends_at": {"year": 2022},
         "description": "Dashboards and churn models in Python/SQL."}],
    "education": [{"school": "University of Tehran", "degree_name": "MSc", "field_of_study": "Computer Science",
                   "starts_at": {"year": 2017}, "ends_at": {"year": 2019}}],
    "skills": ["Python", "LangChain", "LlamaIndex", "SQL", "Gradio"],
    "languages": ["Persian", "English"],
}
my_index = create_vector_database(split_profile_data(my_profile))
print("nodes ok:", verify_embeddings(my_index))
print(generate_initial_facts(my_index))
for q in ["Where did she work before Acme AI?", "What is her favourite food?"]:
    print(f"\nQ: {q}\nA: {answer_user_query(my_index, q).response}")

**Expected / what you should see:** `nodes ok: True`, 3 facts built only from your JSON, *Blue Bank (Data Analyst, 2019–2022)* for the first question and the *I don't know…* sentence for the second. Real LinkedIn data: *Settings → Data privacy → Get a copy of your data*, then convert the CSVs to a dict like this.

### C2 · Icebreakers instead of facts
Add an `ICEBREAKER_TEMPLATE` that writes 3 friendly conversation openers, and a `generate_icebreakers(index)` function.

In [ ]:
from llama_index.core import PromptTemplate
from modules.llm_interface import create_watsonx_llm
ICEBREAKER_TEMPLATE = """Context about a person:\n{context_str}\n
Write 3 short, friendly icebreaker questions I could ask this person at a networking event. Use only the context."""
def generate_icebreakers(index):
    qe = index.as_query_engine(llm=create_watsonx_llm(temperature=0.7), similarity_top_k=config.SIMILARITY_TOP_K,
                               text_qa_template=PromptTemplate(ICEBREAKER_TEMPLATE))
    return qe.query("icebreakers").response
print(generate_icebreakers(index))
# TODO: add a third tab "Icebreakers" to app.py that calls this function.

In [ ]:
# @title ✅ Answer — C2 · Icebreakers tab in the app
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

import gradio as gr, app       # app.py's functions + its active_indices dict
def icebreakers_for_session(session_id):
    if not session_id or session_id not in app.active_indices:
        return "No profile loaded. Please process a LinkedIn profile first."
    return generate_icebreakers(app.active_indices[session_id])

with gr.Blocks(title="LinkedIn Icebreaker Bot") as demo_c2:
    gr.Markdown("# LinkedIn Icebreaker Bot")
    with gr.Tab("Process LinkedIn Profile"):
        url = gr.Textbox(label="LinkedIn Profile URL"); key = gr.Textbox(label="ProxyCurl API Key", type="password")
        mock = gr.Checkbox(label="Use Mock Data", value=True)
        model = gr.Dropdown(["gemini-3.8-flash", "gemini-3.5-flash-lite"], value=config.LLM_MODEL_ID, label="Select LLM Model")
        facts = gr.Textbox(label="Initial Facts", lines=10); sid = gr.Textbox(visible=False)
        gr.Button("Process Profile").click(app.process_profile, [url, key, mock, model], [facts, sid], api_name="process_profile")
    with gr.Tab("Icebreakers"):                                    # ← the new third tab
        ice = gr.Textbox(label="Icebreakers", lines=6)
        gr.Button("Generate icebreakers").click(icebreakers_for_session, [sid], [ice], api_name="icebreakers")
    with gr.Tab("Chat"):
        bot = gr.Chatbot(height=400); q = gr.Textbox(label="Ask a question about the profile")
        q.submit(app.chat_with_profile, [sid, q, bot], [bot], api_name="chat_with_profile")
demo_c2.launch(share=True)     # not blocking in the notebook; demo_c2.close() to stop
# To change app.py itself: paste ICEBREAKER_TEMPLATE + generate_icebreakers into modules/query_engine.py
# and the "Icebreakers" tab block into create_gradio_interface(), using the existing session_id Textbox.

**Expected / what you should see:** a `*.gradio.live` link. *Process Profile* → *Icebreakers* tab → *Generate icebreakers* gives 3 friendly questions about the person; before processing it says *No profile loaded…*.

### C3 · Chunk size and top-k
Try `CHUNK_SIZE` 200 / 500 / 1000 and `SIMILARITY_TOP_K` 2 / 5 and compare answers to the same question.

In [ ]:
for cs in [200, 500, 1000]:
    config.CHUNK_SIZE = cs
    idx = create_vector_database(split_profile_data(profile))
    print(f"\n== chunk_size={cs}\n", answer_user_query(idx, "What was their career progression?").response[:400])
config.CHUNK_SIZE = 500

In [ ]:
# @title ✅ Answer — C3 · Chunk size × top-k
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

q = "What was their career progression?"
for cs in [200, 500, 1000]:
    config.CHUNK_SIZE = cs
    nds = split_profile_data(profile)
    idx = create_vector_database(nds)
    for k in [2, 5]:
        config.SIMILARITY_TOP_K = k               # answer_user_query reads it at call time
        ans = answer_user_query(idx, q).response
        print(f"chunk_size={cs:4d} ({len(nds):2d} nodes) · top_k={k} → {len(ans):4d} chars | {ans[:120]!r}")
config.CHUNK_SIZE, config.SIMILARITY_TOP_K = 500, 5      # restore the lab defaults

**Expected / what you should see:** 6 lines. Smaller chunks = more nodes (≈ 40 / 15 / 8). `top_k=2` with 200-char chunks misses roles; `top_k=5` with 500–1000 gives the fullest timeline. The lab's optional tuning (`CHUNK_SIZE = 400`, `SIMILARITY_TOP_K = 7`) sits in between.

### C4 · Compare Gemini models
Switch models with `change_llm_model` and compare speed and grounding (does it say *I don't know* for missing info?).

In [ ]:
import time
from modules.llm_interface import change_llm_model
for mid in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash"]:
    change_llm_model(mid); t = time.time()
    ans = answer_user_query(index, "What is their favourite food?").response
    print(f"{mid}: {time.time() - t:.1f}s → {ans[:200]}")

In [ ]:
# @title ✅ Answer — C4 · Compare Gemini models (with errors handled)
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

import time
for mid in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash"]:
    change_llm_model(mid); t = time.time()
    try:
        known = answer_user_query(index, "What is this person's current job title?").response
        unknown = answer_user_query(index, "What is their favourite food?").response
        grounded = "don't know" in unknown.lower()
        print(f"{mid:22s} {time.time() - t:5.1f}s · grounded={grounded} · {known[:90]!r}")
    except Exception as e:          # 2.5 access limit on new keys (404/403) or free-tier 429
        print(f"{mid:22s} ⚠️ {type(e).__name__}: {str(e)[:80]}")
    time.sleep(2)                   # be gentle with the free-tier rate limit
change_llm_model("gemini-3.8-flash")

**Expected / what you should see:** one line per model with time, `grounded=True` (it says *I don't know…* for the food question) and the job title. Flash-lite models are fastest. `gemini-2.5-flash` may show ⚠️ on a new API key; that is the 2.5 access limit, not your code.

### C5 · Chat engine with memory
LlamaIndex's `as_chat_engine` keeps the conversation so follow-ups like *"and before that?"* work.

In [ ]:
chat_engine = index.as_chat_engine(chat_mode="context", llm=create_watsonx_llm(), similarity_top_k=3)
print(chat_engine.chat("Where does this person work now?").response)
print(chat_engine.chat("And where did they work before that?").response)
# TODO: use the chat engine in app.py's chat_with_profile so the web chat has memory.

In [ ]:
# @title ✅ Answer — C5 · Chat with memory in the app
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer, then run this cell."]
    raise AnswerNotRun("Answer not run")

import gradio as gr, app
chat_engines = {}                                 # one memory per session id
def chat_with_memory(session_id, user_query, chat_history):
    if not session_id or session_id not in app.active_indices:
        return chat_history + app._turn(user_query, "No profile loaded. Please process a LinkedIn profile first.")
    if session_id not in chat_engines:
        chat_engines[session_id] = app.active_indices[session_id].as_chat_engine(
            chat_mode="context", llm=create_watsonx_llm(), similarity_top_k=3,
            system_prompt="Answer only from the LinkedIn profile context. If it is not there, say you don't know.")
    answer = chat_engines[session_id].chat(user_query).response
    return chat_history + app._turn(user_query, answer)

with gr.Blocks(title="LinkedIn Icebreaker Bot") as demo_c5:
    url = gr.Textbox(label="LinkedIn Profile URL"); key = gr.Textbox(visible=False)
    mock = gr.Checkbox(label="Use Mock Data", value=True); model = gr.Textbox(value=config.LLM_MODEL_ID, visible=False)
    facts = gr.Textbox(label="Initial Facts", lines=6); sid = gr.Textbox(visible=False)
    gr.Button("Process Profile").click(app.process_profile, [url, key, mock, model], [facts, sid], api_name="process_profile")
    bot = gr.Chatbot(height=400); q = gr.Textbox(label="Ask (follow-ups work)")
    q.submit(chat_with_memory, [sid, q, bot], [bot], api_name="chat_with_profile")
demo_c5.launch(share=True)
# In app.py: put chat_engines + this function in place of chat_with_profile's answer_user_query call.

**Expected / what you should see:** in the app, *Where does this person work now?* then *And before that?* — the second answer names the previous employer, because the chat engine keeps the history (the original `chat_with_profile` would not know what *that* refers to).